# Colab Demo: Baseline RAG with Qwen3

This notebook runs baseline RAG with:
- local Colab repo and local virtual environment (fast dependency install)
- Drive-hosted artifacts via a local data symlink
- fixed generator model `Qwen/Qwen3-4B-Instruct-2507`

Outputs:
- On-screen qualitative results (query, evidence, response, claims)
- JSON file under local `outputs/rag_demo_results_<timestamp>.json`
- Optional copy of output JSON to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Local repo for fast environment setup
REPO_DIR = Path('/content/AIST-FYP')
REPO_URL = 'https://github.com/xiashuidaolaoshuren/AIST-FYP.git'
REPO_BRANCH = 'main'

# Drive-hosted artifacts root (same structure as repo data/)
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/data')

# Retrieval strategy matching data/indexes/{STRATEGY}
STRATEGY = 'validation'

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print(f'Using existing local repo: {REPO_DIR}')
else:
    print(f'Cloning repo: {REPO_URL}')
    os.system(f'git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}')

# Critical: baseline_rag.from_config resolves relative paths (data/indexes/...),
# so force cwd to repo root for all subsequent cells.
os.chdir(REPO_DIR)
print('Current working directory:', Path.cwd())

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

print('Installing dependencies...')

# Ensure we start with system Python, not any pre-existing venv
os.environ.pop('VIRTUAL_ENV', None)
original_path = os.environ.get('PATH', '')

repo_dir = Path(REPO_DIR)
uv_project = repo_dir / 'colab' / 'env'
uv_extra = 'evaluation'

# Force uv venv to stay in local repo, never under Drive
os.environ['UV_PROJECT_ENVIRONMENT'] = str(uv_project / '.venv')


def shell_join(parts):
    return ' '.join(shlex.quote(str(p)) for p in parts)


def run(cmd, cwd=None, check=True, stream=True):
    print(f"\n$ {cmd}")
    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(cwd) if cwd is not None else None,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout=''.join(out_lines),
            stderr=''
        )
    else:
        completed = subprocess.run(
            cmd,
            shell=True,
            cwd=str(cwd) if cwd is not None else None,
            text=True,
            capture_output=True,
            check=False
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)

    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed ({completed.returncode}): {cmd}')
    return completed


print('Repo dir:', repo_dir)
print('UV project:', uv_project)
print('UV venv path:', os.environ['UV_PROJECT_ENVIRONMENT'])

# Ensure system Python has pip and uv
run('python -m ensurepip --upgrade', cwd=repo_dir, check=False, stream=True)
run('python -m pip install -U pip wheel setuptools', cwd=repo_dir, check=False, stream=True)
run('python -m pip install -U gradio', cwd=repo_dir, check=False, stream=True)
run('python -m pip install -U uv', cwd=repo_dir, check=False, stream=True)

active_python = 'python'
active_pip_cmd = 'python -m pip'

# Ensure core runtime packages exist in the active environment
runtime_packages = ['transformers', 'pyyaml', 'tqdm', 'sentencepiece', 'accelerate', 'faiss-cpu', 'rank_bm25']
run(f"{active_pip_cmd} install -U " + ' '.join(runtime_packages), cwd=repo_dir, check=False, stream=True)

sync_cmd = shell_join(['uv', 'sync', '--project', uv_project, '--extra', uv_extra])
sync_result = run(sync_cmd, cwd=repo_dir, check=False, stream=True)
print('uv sync return code:', sync_result.returncode)

spacy_model_wheel = (
    'https://github.com/explosion/spacy-models/releases/download/'
    'en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl'
)
spacy_install_cmd = 'python -m spacy download en_core_web_sm'

if sync_result.returncode == 0:
    uv_python = uv_project / '.venv' / 'bin' / 'python'
    run(str(uv_python) + ' -m ensurepip --upgrade', cwd=repo_dir, check=False, stream=True)
    os.environ['PATH'] = f"{uv_python.parent}:{original_path}"

    active_python = str(uv_python)
    active_pip_cmd = f'{active_python} -m pip'
    spacy_install_cmd = shell_join(['uv', 'pip', 'install', '--python', uv_python, spacy_model_wheel])

    print(f'uv sync complete: {uv_project}')
    print('Activated venv bin:', uv_python.parent)
else:
    print('\nuv sync failed. Falling back to pip requirements install...')

    requirements_path = repo_dir / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = shell_join([
        'python', '-m', 'pip', 'install', '--extra-index-url', pytorch_index, '-r', requirements_path
    ])
    fallback_result = run(install_cmd, cwd=repo_dir, check=False, stream=True)

    if fallback_result.returncode != 0:
        print('\nFull requirements install failed. Using Colab-torch-compatible filtered install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = repo_dir / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')

        run(
            'python - <<"PY"\n'
            'import torch\n'
            'import torchvision\n'
            'import torchaudio\n'
            'print("torch", torch.__version__)\n'
            'print("torchvision", torchvision.__version__)\n'
            'print("torchaudio", torchaudio.__version__)\n'
            'PY',
            cwd=repo_dir,
            check=False,
            stream=True
        )
        run(shell_join(['python', '-m', 'pip', 'install', '-r', temp_req]), cwd=repo_dir, stream=True)

# Install spaCy model with the chosen environment
run(spacy_install_cmd, cwd=repo_dir, stream=True)

# Quick sanity checks for downstream demo (against active interpreter)
run(f"{active_python} -c \"import torch, transformers, yaml; print(torch.__version__)\"", cwd=repo_dir, stream=True)
print('Dependency installation step complete.')

In [ ]:
import torch

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA is not available. Switch Colab runtime to GPU before running the demo.')

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import shutil
import time

# Use the strategy selected in the setup cell above (do not override here)
PROJECT_ROOT = Path(REPO_DIR)
LOCAL_DATA_PATH = PROJECT_ROOT / 'data'
DRIVE_INDEXES_ROOT = DRIVE_DATA_ROOT / 'indexes'
LEGACY_DRIVE_INDEX_ROOT = DRIVE_DATA_ROOT / 'index'

required_files = [
    DRIVE_INDEXES_ROOT / f'{STRATEGY}/faiss.index',
    DRIVE_INDEXES_ROOT / f'{STRATEGY}/metadata.pkl',
    DRIVE_INDEXES_ROOT / f'{STRATEGY}/bm25_index.pkl',
    DRIVE_DATA_ROOT / f'processed/wiki_chunks_{STRATEGY}.jsonl',
]

missing = []
for p in tqdm(required_files, desc='Checking Drive artifacts', unit='file'):
    if not p.exists():
        missing.append(str(p))

if missing:
    extra_hint = ''
    if LEGACY_DRIVE_INDEX_ROOT.exists() and not DRIVE_INDEXES_ROOT.exists():
        extra_hint = (
            f"\nDetected legacy folder: {LEGACY_DRIVE_INDEX_ROOT}. "
            "Pipeline expects '/data/indexes/'. Rename/move it to '/data/indexes/' on Drive."
        )
    raise FileNotFoundError(
        'Missing required artifacts in DRIVE_DATA_ROOT:\n' + '\n'.join(missing) + extra_hint
    )

# Safety: never mutate anything that resolves under /content/drive.
# If git pull created a local ./data directory, back it up locally and replace with symlink.
if LOCAL_DATA_PATH.exists() or LOCAL_DATA_PATH.is_symlink():
    if LOCAL_DATA_PATH.is_symlink():
        current_target = LOCAL_DATA_PATH.resolve()
        if current_target != DRIVE_DATA_ROOT.resolve():
            LOCAL_DATA_PATH.unlink()
            LOCAL_DATA_PATH.symlink_to(DRIVE_DATA_ROOT, target_is_directory=True)
            print(f'Updated data symlink: {LOCAL_DATA_PATH} -> {DRIVE_DATA_ROOT}')
        else:
            print(f'Data symlink already correct: {LOCAL_DATA_PATH} -> {current_target}')
    else:
        resolved_local_data = LOCAL_DATA_PATH.resolve()
        if str(resolved_local_data).startswith('/content/drive'):
            raise RuntimeError(
                f'Unsafe to mutate Drive-mounted path: {resolved_local_data}. '
                'Please stop and recover Drive data first.'
            )

        backup_path = PROJECT_ROOT / f'data.local_backup_{int(time.time())}'
        shutil.move(str(LOCAL_DATA_PATH), str(backup_path))
        print(f'Moved local data directory to backup: {backup_path}')
        LOCAL_DATA_PATH.symlink_to(DRIVE_DATA_ROOT, target_is_directory=True)
        print(f'Created data symlink: {LOCAL_DATA_PATH} -> {DRIVE_DATA_ROOT}')
else:
    LOCAL_DATA_PATH.symlink_to(DRIVE_DATA_ROOT, target_is_directory=True)
    print(f'Created data symlink: {LOCAL_DATA_PATH} -> {DRIVE_DATA_ROOT}')

local_strategy_index = LOCAL_DATA_PATH / 'indexes' / STRATEGY
if not local_strategy_index.exists():
    raise FileNotFoundError(
        f'Expected strategy index path missing after symlink setup: {local_strategy_index}'
    )

print('Artifact preflight complete for strategy:', STRATEGY)
print('Local strategy index path:', local_strategy_index)

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = Path(REPO_DIR) / 'config.yaml'
COLAB_CONFIG = Path(REPO_DIR) / 'config.colab.qwen3.yaml'

with open(BASE_CONFIG, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['models']['generator'] = 'Qwen/Qwen3-4B-Instruct-2507'
cfg['processing']['device'] = 'cuda'
cfg['generation']['load_in_8bit'] = True
cfg['generation']['max_input_tokens'] = 4096
cfg['generation']['enable_thinking'] = False

# Safety defaults for Colab stability (adjust as needed)
cfg['generation']['max_new_tokens'] = 256
cfg['retrieval']['top_k'] = 5

# Generator-only demo mode: disable verifier and mitigation
cfg.setdefault('verification', {})
cfg['verification']['enabled'] = False

verification_modules = cfg['verification'].get('modules')
if isinstance(verification_modules, dict):
    for key in ['intrinsic', 'grounded', 'nli', 'self_agreement']:
        if key in verification_modules:
            verification_modules[key] = False

cfg.setdefault('mitigation', {})
cfg['mitigation']['enabled'] = False

for section in ['reranker', 'filter', 'reprompt', 'router']:
    sub = cfg['mitigation'].get(section)
    if isinstance(sub, dict) and 'enabled' in sub:
        sub['enabled'] = False

with open(COLAB_CONFIG, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Wrote runtime config:', COLAB_CONFIG)
print('Generator model:', cfg['models']['generator'])
print('load_in_8bit:', cfg['generation']['load_in_8bit'])
print('max_input_tokens:', cfg['generation'].get('max_input_tokens'))
print('Generator-only mode: verification.enabled =', cfg['verification']['enabled'])
print('Generator-only mode: mitigation.enabled =', cfg['mitigation']['enabled'])

In [ ]:
import os
import sys
import json
import time
from datetime import datetime
from pathlib import Path
import numpy as np

# Ensure relative paths in pipeline resolve from repo root
os.chdir(REPO_DIR)
print('Current working directory:', Path.cwd())

sys.path.insert(0, str(REPO_DIR))

from src.pipelines import BaselineRAGPipeline

def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_serializable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj

# Preflight: verify the expected strategy artifacts are visible from current cwd
index_path = Path(f'data/indexes/{STRATEGY}/faiss.index')
metadata_path = Path(f'data/indexes/{STRATEGY}/metadata.pkl')
bm25_path = Path(f'data/indexes/{STRATEGY}/bm25_index.pkl')
print('Preflight index path:', index_path.resolve())
print('Exists faiss.index:', index_path.exists())
print('Exists metadata.pkl:', metadata_path.exists())
print('Exists bm25_index.pkl:', bm25_path.exists())

if not index_path.exists():
    raise FileNotFoundError(
        f'Missing index at {index_path} from cwd={Path.cwd()}. '\
        f'Run the artifact symlink cell first and verify STRATEGY={STRATEGY} exists on Drive.'
    )

pipeline = BaselineRAGPipeline.from_config(
    config_path=str(Path(REPO_DIR) / 'config.colab.qwen3.yaml'),
    strategy=STRATEGY
)

sample_queries = [
    'What is artificial intelligence?',
    'What are the main types of machine learning?',
    'What is machine learning? How does it differ from traditional programming?',
    'What is the capital of Canada and what province is it in?',
    'What caused the fall of the Western Roman Empire?'
]

all_results = []
for i, query in enumerate(sample_queries, start=1):
    t0 = time.time()
    result = pipeline.run(query, top_k=5)
    elapsed = time.time() - t0

    all_results.append({
        'query_index': i,
        'query': query,
        'timestamp': datetime.now().isoformat(),
        'latency_sec': round(elapsed, 3),
        'result': make_json_serializable(result),
    })

print(f'Completed {len(all_results)} queries')

In [ ]:
for item in all_results:
    print('=' * 100)
    print(f"Query {item['query_index']}: {item['query']}")
    print('-' * 100)

    result = item['result']
    retrieval = result.get('retrieval_metadata', {})
    pairs = result.get('claim_evidence_pairs', [])

    print('Latency (sec):', item.get('latency_sec'))
    print('Retrieved chunks:', retrieval.get('num_retrieved'))
    print('Top score:', retrieval.get('top_score'))

    print('Draft response:')
    print(result.get('draft_response', ''))

    print('Claims extracted:', len(pairs))
    if pairs:
        first_pair = pairs[0]
        spans = first_pair.get('evidence_spans', [])
        print('Top evidence candidates (up to 2):')
        for span in spans[:2]:
            text = span.get('text', '')
            print(f"- {span.get('doc_id')}#{span.get('sent_id')}: {text[:180]}")

    print()

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import shutil
from dataclasses import asdict, is_dataclass

def serialize_to_dict(obj):
    """
    Recursively convert dataclass instances and other objects to JSON-serializable dicts.
    Handles nested dataclasses, lists, and dicts.
    """
    if is_dataclass(obj) and not isinstance(obj, type):
        # Convert dataclass to dict
        return asdict(obj)
    elif isinstance(obj, dict):
        # Recursively process dict values
        return {k: serialize_to_dict(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        # Recursively process list items
        return [serialize_to_dict(item) for item in obj]
    elif hasattr(obj, 'to_dict'):
        # Use to_dict() method if available
        return obj.to_dict()
    else:
        # Return as-is for primitive types
        return obj

def sanitize_generation_metadata_for_demo(meta):
    """Drop heavy token payloads while preserving key schema for demo artifacts."""
    if not isinstance(meta, dict):
        return meta

    sanitized = dict(meta)
    sanitized.pop('token_ids', None)
    sanitized['logits'] = []
    sanitized['scores'] = []

    sub_answer_meta = sanitized.get('sub_answer_metadata')
    if isinstance(sub_answer_meta, list):
        updated = []
        for entry in sub_answer_meta:
            if not isinstance(entry, dict):
                updated.append(entry)
                continue

            entry_copy = dict(entry)
            nested_meta = entry_copy.get('metadata')
            if isinstance(nested_meta, dict):
                entry_copy['metadata'] = sanitize_generation_metadata_for_demo(nested_meta)
            updated.append(entry_copy)
        sanitized['sub_answer_metadata'] = updated

    return sanitized

def sanitize_demo_result(result_obj):
    result_dict = serialize_to_dict(result_obj)
    if isinstance(result_dict, dict) and isinstance(result_dict.get('generator_metadata'), dict):
        result_dict['generator_metadata'] = sanitize_generation_metadata_for_demo(
            result_dict['generator_metadata']
        )
    return result_dict

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
outputs_dir = Path(REPO_DIR) / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
output_file = outputs_dir / f'rag_demo_results_{ts}.json'

# Keep the same top-level shape as scripts/demo_baseline_rag.py
json_results = []
for item in all_results:
    result_dict = {
        'query_index': item['query_index'],
        'query': item['query'],
        'timestamp': item['timestamp'],
        'result': sanitize_demo_result(item['result']),
    }
    json_results.append(result_dict)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(json_results, f, ensure_ascii=False, indent=2)

print('Saved locally:', output_file)

# Optional lightweight copy to Drive
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/AIST-FYP/outputs')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
drive_output_file = DRIVE_OUTPUT_DIR / output_file.name
shutil.copy2(output_file, drive_output_file)
print('Copied to Drive:', drive_output_file)

## Troubleshooting

- If you hit OOM, reduce `cfg['generation']['max_new_tokens']` to 128 and run fewer queries.
- If artifacts are missing, verify `DRIVE_DATA_ROOT` and selected `STRATEGY`.
- If model load fails, re-check GPU runtime and rerun setup cells.
- This notebook is configured for generator-only demo mode (`verification` and `mitigation` disabled in runtime config).
- Default generator is `Qwen/Qwen3-4B-Instruct-2507` with `load_in_8bit=True` for Colab stability.
- This notebook expects a local repo at `/content/AIST-FYP` and a Drive artifact tree at `DRIVE_DATA_ROOT`.
- If symlink creation fails, ensure `PROJECT_ROOT/data` is not locked and rerun the artifact cell.
- Saved demo artifacts intentionally exclude heavy token payloads: `generator_metadata.token_ids` is removed, and `generator_metadata.logits`/`scores` are stored as empty arrays for schema compatibility.

In [ ]:
# Drive index diagnostics (read-only)
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/data')
idx_plural = drive_data / 'indexes'
idx_singular = drive_data / 'index'
strategy_path = idx_plural / STRATEGY

print('Drive data root:', drive_data)
print('Exists data root:', drive_data.exists())
print('Exists indexes/ :', idx_plural.exists())
print('Exists index/   :', idx_singular.exists())
print('Selected STRATEGY:', STRATEGY)
print('Strategy path:', strategy_path)
print('Exists strategy path:', strategy_path.exists())

expected = [
    strategy_path / 'faiss.index',
    strategy_path / 'metadata.pkl',
    strategy_path / 'bm25_index.pkl',
]
for p in expected:
    print(f'- {p.name}:', p.exists())

if idx_singular.exists() and not idx_plural.exists():
    print('\n⚠️  Found /data/index but missing /data/indexes.')
    print('    Pipeline expects /data/indexes/{STRATEGY}/...')


## 7. Launch Full Pipeline UI with Hallucination Detection

The previous cells demonstrated baseline RAG generation. This cell launches the **full pipeline** that adds hallucination detection and mitigation on top of RAG:

**What the UI includes:**
- **Interactive Chat**: Query the RAG system and get instant results
- **Color-Coded Claims**: Each claimed fact is highlighted based on confidence:
  - 🟢 **Green (Supported)**: High confidence, grounded in evidence
  - 🟡 **Yellow (Low Confidence)**: Uncertain or insufficient evidence  
  - 🔴 **Red (Contradictory)**: Conflicts with retrieved evidence
- **Per-Claim Metrics**: Detailed table showing verification scores for each claim
- **Evidence Inspector**: View which evidence chunks support each claim

**How it works:**
1. Your query is sent to the RAG pipeline (retrieval + generation)
2. Claims are extracted and verified using 4 signals:
   - **Entropy**: Model's token-level confidence
   - **Coverage**: Named entities/numbers found in evidence
   - **NLI**: Natural Language Inference (contradiction detection)
   - **Self-Consistency**: Agreement across multiple model samples
3. A rule-based aggregator combines signals into final verdicts
4. Optional mitigation: re-ranking evidence or filtering contradictory claims

**Note:** The UI may take 1-2 minutes to initialize (loading the generator model + verifiers). Once running, type your query in the text box and click submit.

Run the cell below to launch the interactive interface:

In [ ]:
import sys
import subprocess
from pathlib import Path

# Check faiss in current notebook kernel
try:
    import faiss
    print('Kernel faiss version:', faiss.__version__)
    print('Kernel faiss import successful')
except Exception as exc:
    print('Kernel faiss import failed:', exc)

# Check sentence_transformers in current notebook kernel
try:
    from sentence_transformers import __version__ as st_version
    print('Kernel sentence_transformers version:', st_version)
    print('Kernel sentence_transformers import successful')
except Exception as exc:
    print('Kernel sentence_transformers import failed:', exc)

# Check faiss and sentence_transformers in launcher interpreter (used by Gradio launch cell)
repo_dir = Path(REPO_DIR)
venv_python = repo_dir / 'colab' / 'env' / '.venv' / 'bin' / 'python'
launcher_python = str(venv_python) if venv_python.exists() else sys.executable

print('Launcher python:', launcher_python)

# Check faiss
faiss_check_cmd = [
    launcher_python,
    '-c',
    "import faiss; print('Launcher faiss version:', faiss.__version__)"
]
faiss_check = subprocess.run(faiss_check_cmd, text=True, capture_output=True)

if faiss_check.returncode == 0:
    print(faiss_check.stdout)
    print('Launcher faiss import successful')
else:
    print('Launcher faiss import failed. Installing faiss-cpu into launcher environment...')
    print(faiss_check.stderr)
    install = subprocess.run(
        [launcher_python, '-m', 'pip', 'install', '-U', 'faiss-cpu'],
        text=True,
        capture_output=True,
        check=False,
    )
    if install.stdout:
        print(install.stdout)
    if install.stderr:
        print(install.stderr)

    faiss_recheck = subprocess.run(faiss_check_cmd, text=True, capture_output=True)
    if faiss_recheck.returncode != 0:
        raise RuntimeError(
            'faiss still unavailable in launcher interpreter. '\
            f'\nSTDERR:\n{faiss_recheck.stderr}'
        )

    print(faiss_recheck.stdout)
    print('Launcher faiss import successful after installation')

# Check sentence_transformers
st_check_cmd = [
    launcher_python,
    '-c',
    "from sentence_transformers import __version__; print('Launcher sentence_transformers version:', __version__)"
]
st_check = subprocess.run(st_check_cmd, text=True, capture_output=True)

if st_check.returncode == 0:
    print(st_check.stdout)
    print('Launcher sentence_transformers import successful')
else:
    print('Launcher sentence_transformers import failed. Installing sentence-transformers into launcher environment...')
    print(st_check.stderr)
    install = subprocess.run(
        [launcher_python, '-m', 'pip', 'install', '-U', 'sentence-transformers'],
        text=True,
        capture_output=True,
        check=False,
    )
    if install.stdout:
        print(install.stdout)
    if install.stderr:
        print(install.stderr)

    st_recheck = subprocess.run(st_check_cmd, text=True, capture_output=True)
    if st_recheck.returncode != 0:
        raise RuntimeError(
            'sentence_transformers still unavailable in launcher interpreter. '\
            f'\nSTDERR:\n{st_recheck.stderr}'
        )

    print(st_recheck.stdout)
    print('Launcher sentence_transformers import successful after installation')

In [ ]:
# Check and install critical dependencies in launcher interpreter.
# Test all together to catch missing packages, then install all at once if needed.
critical_packages_check_cmd = [
    python_exec,
    '-c',
    (
        "import faiss, gradio, accelerate; "
        "from sentence_transformers import __version__ as st_v; "
        "from rank_bm25 import BM25Okapi; "
        "print('faiss', faiss.__version__); "
        "print('sentence_transformers', st_v); "
        "print('rank_bm25 available'); "
        "print('gradio', gradio.__version__); "
        "print('accelerate', accelerate.__version__)"
    ),
]
critical_packages_check = subprocess.run(critical_packages_check_cmd, text=True, capture_output=True)

if critical_packages_check.returncode != 0:
    print('Missing critical packages in launcher interpreter; installing...')
    print(critical_packages_check.stderr)
    install = subprocess.run(
        [
            python_exec,
            '-m',
            'pip',
            'install',
            '-U',
            'faiss-cpu',
            'sentence-transformers',
            'rank_bm25',
            'gradio',
            'accelerate',
        ],
        text=True,
        capture_output=True,
        check=False,
    )
    if install.stdout:
        print(install.stdout)
    if install.stderr:
        print(install.stderr)

    critical_packages_check = subprocess.run(critical_packages_check_cmd, text=True, capture_output=True)
    if critical_packages_check.returncode != 0:
        raise RuntimeError(
            'Critical packages still unavailable in launcher interpreter after install.'
            f"\nSTDERR:\n{critical_packages_check.stderr}"
        )

print(critical_packages_check.stdout)
print('Launcher critical packages import successful')